In [1]:
from top2vec import Top2Vec
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotesProcessed.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def top2vec_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time()
    top2vec_model = Top2Vec(
        texts,
        embedding_model='all-MiniLM-L6-v2',
        speed="learn"
    )
    cluster_topics = (top2vec_model.get_topics())[0]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    top2vec_analysis(nurse_notes[key]['Note'])
    all_texts.extend(nurse_notes[key]['Note'])

-----------P1-----------


2026-02-02 10:24:38,998 - top2vec - INFO - Pre-processing documents for training


Number of texts: 600


/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-02-02 10:24:39,170 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Coherence: 0.47632452151994015
Diversity: 0.475
Inverse Redundancy: 0.4833333333333333
Time (seconds): 29.194651126861572
----- Cluster Topics -----
['resident' 'med' 'wheelchair' 'laxative' 'sit' 'prescribe' 'form'
 'administer' 'medication' 'care']
['bed' 'sleep' 'resident' 'med' 'night' 'comfortable' 'care' 'sit'
 'wheelchair' 'morning']
['bed' 'sleep' 'medication' 'med' 'night' 'settle' 'toilete' 'tts'
 'comfortable' 'resident']
['bed' 'sit' 'sleep' 'medication' 'med' 'wheelchair' 'overnight' 'night'
 'morning' 'prescribe']
Number of Topics: 4
-----------P9-----------
Number of texts: 624


2026-02-02 10:32:30,309 - top2vec - INFO - Pre-processing documents for training
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-02-02 10:32:30,396 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-02 10:32:46,050 - top2vec - INFO - Creating joint document/word embedding
2026-02-02 10:33:05,461 - top2vec - INFO - Creating lower dimension embedding of documents
2026-02-02 10:33:09,856 - top2vec - INFO - Finding dense areas of documents
2026-02-02 10:33:09,981 - top2vec - INFO - Finding topics


Coherence: 0.34334388385088044
Diversity: 0.475
Inverse Redundancy: 0.5
Time (seconds): 40.32445693016052
----- Cluster Topics -----
['resident' 'med' 'meds' 'compliant' 'medication' 'care' 'administer'
 'concern' 'form' 'maintain']
['adls' 'compliant' 'meds' 'med' 'resident' 'safety' 'settle' 'maintain'
 'administer' 'medication']
['sensor' 'mat' 'safety' 'toilete' 'compliant' 'resident' 'assist' 'need'
 'settle' 'chart']
['resident' 'sensor' 'med' 'compliant' 'safety' 'mat' 'form' 'meds' 'care'
 'concern']
Number of Topics: 4


In [7]:
top2vec_analysis(all_texts)

2026-02-02 10:33:18,515 - top2vec - INFO - Pre-processing documents for training


Number of texts: 12191


/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-02-02 10:33:20,647 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-02 10:33:30,785 - top2vec - INFO - Creating joint document/word embedding
2026-02-02 10:34:29,841 - top2vec - INFO - Creating lower dimension embedding of documents
2026-02-02 10:35:35,894 - top2vec - INFO - Finding dense areas of documents
2026-02-02 10:35:37,404 - top2vec - INFO - Finding topics


Coherence: 0.32940147840510037
Diversity: 0.13303571428571428
Inverse Redundancy: 0.7138513513513514
Time (seconds): 139.14962196350098
----- Cluster Topics -----
['hospital' 'bed' 'appointment' 'resident' 'sleep' 'comfort' 'slept' 'med'
 'asleep' 'nurse']
['resident' 'appointment' 'hospital' 'form' 'med' 'assessment' 'referral'
 'assistance' 'care' 'compliant']
['resident' 'hospital' 'appointment' 'assistance' 'med' 'apply'
 'wheelchair' 'compliant' 'form' 'nurse']
['adls' 'compliant' 'resident' 'meds' 'med' 'hospital' 'safety'
 'appointment' 'ensure' 'settle']
['asleep' 'hospital' 'skin' 'sleep' 'resident' 'comfort' 'awake' 'slept'
 'appointment' 'care']
['bed' 'medication' 'appointment' 'sleep' 'hospital' 'nurse' 'meds' 'med'
 'slept' 'medicine']
['sleep' 'slept' 'asleep' 'awake' 'care' 'relax' 'bed' 'alarm' 'rest'
 'stay']
['hospital' 'appointment' 'resident' 'med' 'referral' 'nurse' 'attend'
 'form' 'assistance' 'medicine']
['laxative' 'bowel' 'resident' 'hospital' 'toilet' 'appoi